In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# unit and sanity checks!
"""--------------------------------------------"""
# check for temporal autocorrelations by ensuring no encoding after shuffling trial data.
# incorporate as a sanity check to pass for all sessions

# > check the fits for different regularization constants
# define responsive

# one regressor
# cvr2 with strategy model params
# add time
"""--------------------------------------------"""

## init

In [ ]:
import numpy as np
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id)
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
)
se.plot_cvr2()
se.plot_dr2()
se.plot_bound_r2()

## weight correlation

In [ ]:
# between mb and mf blocks, do neurons encode
# reward prediction error (MF), q-learner, need to get q-value
# i might expect mb blocks to have a stronger encoding of the block identity
# mf and trial history?
# build in interaction terms and see what pops out in the cvr2/dr2 plots
# cvr2 across time
# -> seeing when encoding emerges

In [ ]:
from core.data import tv_vals
from core.viz import plot_kdes

weight_diff = {}
for regr in encoder.tv_keys:
    if regr != "response_prev":
        regr_ = f"{regr}_{tv_vals[regr][0]}"
        weight_diff[regr_] = (
            encoder_mb.encoder_weights[:, encoder.dm_idxs[regr_]]
            - encoder_mf.encoder_weights[:, encoder.dm_idxs[regr_]]
        )

plot_kdes(weight_diff)

## pca based on weights

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(encoder.encoder_weights)

In [ ]:
cum_var = np.array(
    [
        sum(pca.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(cum_var)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.show()

In [ ]:
n = np.where(cum_var >= 0.9)[0][0]
pca = PCA(n_components=n).fit(encoder.encoder_weights)
weights_lowd = pca.transform(encoder.encoder_weights)

In [ ]:
weights_lowd[:, :3]

In [ ]:
plt.figure()
plt.scatter(weights_lowd[:, 0], weights_lowd[:, 1], alpha=0.5, s=0.5)
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection="3d")

ax.scatter(xs=weights_lowd[:, 0], ys=weights_lowd[:, 1], zs=weights_lowd[:, 2], s=0.3)
plt.show()

## kmeans clustering

In [ ]:
# pick ideal num clusters, plot weight matrices for each cluster
# * only using tv weights

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

n_clusters = range(2, encoder.num_units // 2)
seeds = range(10)
silhouettes = np.zeros((len(n_clusters), len(seeds)))

for i, n in enumerate(n_clusters):
    for j, seed in enumerate(seeds):
        km = KMeans(n_clusters=n, random_state=seed)
        cluster_labels = km.fit_predict(encoder.encoder_weights[:, encoder.num_tents :])
        silhouettes[i][j] = silhouette_score(
            encoder.encoder_weights[:, encoder.num_tents :], cluster_labels
        )

n = n_clusters[np.argmax(silhouettes.mean(axis=1))]
print("cluster w/ best avg. silhouette score: ", n)

In [ ]:
silhouettes_mean = silhouettes.mean(axis=1)
silhouettes_std = silhouettes.std(axis=1)

plt.figure(tight_layout=True)
plt.plot(n_clusters, silhouettes_mean)
plt.fill_between(
    n_clusters,
    silhouettes_mean - silhouettes_std,
    silhouettes_mean + silhouettes_std,
    alpha=0.4,
)
plt.xlabel("n. clusters")
plt.ylabel("silhouette score (mean)")
plt.show()

In [ ]:
km = KMeans(n_clusters=n, random_state=2)
km_labels = km.fit_predict(encoder.encoder_weights[:, encoder.num_tents :])

In [ ]:
n_cols = encoder.encoder_weights.shape[1] - encoder.num_tents
cluster_sizes = [(km_labels == i).sum() for i in range(n)]
max_rows = max(cluster_sizes)

px = 0.03  # inches per pixel — tune this to taste

fig, axes = plt.subplots(
    ncols=n,
    nrows=1,
    figsize=(n * n_cols * px, max_rows * px),
)

for i, ax in enumerate(axes.flat):
    data = encoder.encoder_weights[np.where(km_labels == i)[0], encoder.num_tents :]
    im = ax.imshow(data, vmin=-2, vmax=2, cmap="coolwarm", aspect="auto")

    ax.set_box_aspect(
        data.shape[0] / n_cols
    )  # forces axes height ∝ n_rows, width unchanged
    ax.set_anchor("N")  # top-align so smaller clusters don't float to center
    ax.set_xticks([])
    ax.set_yticks([])

## r2 comp between regions and strategies 

scatter version is in .verify()

In [ ]:
import numpy as np

np.mean(encoder_mb.scores["encoder"] > encoder_mf.scores["encoder"])

In [ ]:
encoder_mb_cond = StrategyEncoder(
    subj_id, sess_id, strategy_filter="mb", balance_strategy=True, cond_balance=True
)
encoder_mb_count = StrategyEncoder(
    subj_id, sess_id, strategy_filter="mb", balance_strategy=True, cond_balance=False
)
encoder_mf_count = StrategyEncoder(
    subj_id, sess_id, strategy_filter="mf", balance_strategy=True, cond_balance=False
)
encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")

In [ ]:
encoder_mf.scores["encoder"] == encoder_mf_count.scores["encoder"]

In [ ]:
encoder_mb_cond.verify()
encoder_mb_count.verify()
encoder_mf_count.verify()
encoder_mb.verify()

In [ ]:
from core.viz import plot_scatter

plot_scatter(
    x=encoder_mb.scores["encoder"],
    y=encoder_mf.scores["encoder"],
    xlabel="mb",
    ylabel="mf",
    add_unity=True,
    add_lr=True,
)
plot_scatter(
    x=encoder_mb_count.scores["encoder"],
    y=encoder_mf_count.scores["encoder"],
    xlabel="mb",
    ylabel="mf",
    add_unity=True,
    add_lr=True,
)

In [ ]:
# r2 between DMS and DLS
from core.viz import plot_kdes
from utils.colors import colors_region

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
# scores
scores = {
    k: {
        f"{reg}, {model}": encoder_.scores[model][encoder_.reg_idxs[reg]]
        for reg in encoder_.regions
        for model in ["baseline", "encoder"]
    }
    for k, encoder_ in encoders.items()
}

# styles
linestyles = {"baseline": "--", "encoder": "-"}
linewidths = {"baseline": 0.5, "encoder": 1}
styles = {
    f"{reg}, {model}": {
        "linestyle": linestyles[model],
        "linewidth": linewidths[model],
        "color": colors_region[reg],
    }
    for reg in encoder.regions
    for model in ["baseline", "encoder"]
}

for k, scores_ in scores.items():
    plot_kdes(
        scores_,
        label=rf"$r^2$, {k}",
        xlim=(-0.25, 1),
        add_means=False,
        line_kwargs=styles,
    )

In [ ]:
from core.data import colors_strategy

scores_baseline = {k: encoder_.scores["baseline"] for k, encoder_ in encoders.items()}
styles = {
    "full": {"color": "#444444", "linewidth": 1},
    "mb": {"color": colors_strategy["mb"], "linestyle": "--"},
    "mf": {"color": colors_strategy["mf"], "linestyle": "--"},
}
plot_kdes(scores_baseline, label=r"$r^2$, baseline", line_kwargs=styles)

## weight comp between regions and strategies

### kde

In [ ]:
# norm spike counts
from core.viz import plot_kde_row

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

sc_mean = {
    k: {
        reg: encoder_.robs[:, encoder_.reg_idxs[reg]].mean(axis=0)
        for reg in encoder_.regions
    }
    for k, encoder_ in encoders.items()
}

plot_kde_row(sc_mean)

In [ ]:
from core.viz import plot_kde_row
from core.data import colors_strategy
from utils.paths import FIGURES_DIR

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

# styles
colors_region = {"DMS": "#2383DC", "DLS": "#3DC1E2"}
styles_strategy = {f"{reg}": {"color": colors_region[reg]} for reg in encoder.regions}
styles_reg = {f"{k}": {"color": colors_strategy[k]} for k in encoders}

# iterate through all regressors and save
for regressor in encoder.dm_names:
    if "tents" not in regressor:
        # weights
        weights_strategy = {
            k: {
                f"{reg}": encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for reg in encoder_.regions
            }
            for k, encoder_ in encoders.items()
        }

        weights_reg = {
            reg: {
                f"{k}": encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for k, encoder_ in encoders.items()
            }
            for reg in encoder.regions
        }

        # plot
        for k, weights, styles in zip(
            ["strategy", "region"],
            [weights_strategy, weights_reg],
            [styles_strategy, styles_reg],
        ):
            fig, _ = plot_kde_row(
                weights, styles, title=rf"$\beta$ {regressor}", add_means=False
            )

            fpath = FIGURES_DIR / "lite" / subj_id / sess_id / "bweight"
            fpath.mkdir(parents=True, exist_ok=True)
            fig.savefig(fpath / f"{regressor}_{k}.png", dpi=300, bbox_inches="tight")

            plt.close(fig)

### scatter, hist 2d, contour

In [ ]:
import numpy as np
from core.data import tv_vals
from core.viz import plot_scatter, plot_hist2d, plot_contour, plot_2d_row
from utils.viz_utils import save_fig

fpath = FIGURES_DIR / "lite" / subj_id / sess_id / "bweight"

for regr in encoder.tv_keys:
    vals = [tv_vals[regr][0]] if not regr == "response_prev" else tv_vals[regr]

    for val in vals:
        regressor = f"{regr}_{val}"
        weights = {
            reg: [
                encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for encoder_ in [encoder_mb, encoder_mf]
            ]
            for reg in encoder.regions
        }

        # scatter
        fig, ax = plot_2d_row(
            plot_scatter,
            weights,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            add_unity=True,
            add_lr=True,
        )

        save_fig(fig, fpath / "scatter", f"{regressor}.png")

        # mn/mx
        mn = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).min()
        )
        mx = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).max()
        )

        # hist2d
        fig, ax = plot_2d_row(
            plot_hist2d,
            weights,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            sharey=True,
            sharex=True,
        )

        save_fig(fig, fpath / "hist2d", f"{regressor}.png")

        # contour
        fig, ax = plot_2d_row(
            plot_contour,
            weights,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            fill=False,
            sharey=True,
            sharex=True,
        )

        save_fig(fig, fpath / "contour", f"{regressor}.png")